In [ ]:
import h5py
import numpy as np

import scipy
import scipy.fft as fft
import scipy.constants as constants

from IPython.display import clear_output
from skimage.morphology import convex_hull_image

from helper_functions import (sphere_idx, write_text, radial, interpolated_intercepts, fourier_shell_correlation)
import spimage

import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

import seaborn as sns
sns.set_theme()

import sys, time
import os, os.path

pi = scipy.constants.pi
e = constants.elementary_charge
h = constants.Planck
c = constants.speed_of_light

<h2> Loading 3D model to phase </h2> 

In [ ]:
subBg = False

saveCorr = False
diagPlotEMC = True
diagPlotSIM = False
plotAutoCorr = False

emc_file = 'emc/protein_water_ds_4x_fin_0001/data_500k/prot_only_0/output_130.h5'

with h5py.File(emc_file, "r") as f_ptr:
    I_emc = np.squeeze(f_ptr['intens'][:])
    W_emc = np.squeeze(f_ptr['inter_weight'][:])

I_emc = I_emc[:-1,:-1,:-1]
W_emc = W_emc[:-1,:-1,:-1]

if subBg:
    emc_bg_file = 'emc/water_only_ds_4x/data_500k/wat_only_0_2/output_130.h5'
    with h5py.File(emc_bg_file,'r') as f_bg:
        I_emc_bg = np.squeeze(f_bg['intens'][:])

    I_emc_bg = I_emc_bg[:-1,:-1,:-1]
    frame = I_emc - I_emc_bg
else:
    frame = I_emc

# Condor model 
#condor_file = '3D_fourier_models/3d_groel_agipd_9_kev_denss_dsf_4x_fin.h5'
#with h5py.File(condor_file, "r") as f_cond:
#    I_emc = f_cond['3Dparticle/intensity'][:]

frame = I_emc

center = frame.shape[0]//2

mask_emc = W_emc.astype(np.bool_)
frac_emc_good = mask_emc.sum() / (mask_emc.shape[0]**3)

mask_pos = frame >= 0.0
mask_all = mask_emc & mask_pos

file = emc_file
npats = file.split(sep='/')[1].split(sep='_')[1]
sType = file.split(sep="/")[1]
pType = 'emc_'+file.split(sep='/')[3]
print(f'Phasing {pType} model ({npats} patterns)!')
    
if diagPlotEMC:
    frame_scaling = 1.0
    test_slice = center
    
    xy_obj = frame[test_slice,:,:] ** frame_scaling
    xz_obj = frame[:,test_slice,:] ** frame_scaling
    yz_obj = frame[:,:,test_slice] ** frame_scaling

    fig_handle = plt.figure(constrained_layout = True, dpi = 200)
    fig_handle.patch.set_facecolor('gray')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
    cm = 'viridis'
    max_v = 0.5

    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_0.set_xticks([])
    ax_0.set_yticks([])
    minv, maxv = im_0.get_clim()
    ax_0.set_title(f'XY-{test_slice}',weight='bold',fontsize=6) 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,shrink=0.35) 
    c_bar_0.set_ticks([minv,maxv*0.5,maxv])

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim()
    ax_1.set_title(f'XZ-{test_slice}',weight='bold',fontsize=6)

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_2.set_xticks([])
    ax_2.set_yticks([])
    minv, maxv = im_2.get_clim()
    ax_2.set_title(f' YZ-{test_slice}',weight='bold',fontsize=6);
    
    fig_handle = plt.figure(constrained_layout = True, dpi = 200)
    fig_handle.patch.set_facecolor('gray')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
    cm = 'gray'
    max_v = 1.0

    xy_obj = mask_emc[:,:,test_slice]
    xz_obj = mask_emc[:,test_slice,:]
    yz_obj = mask_emc[test_slice,:,:]
    
    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_0.set_xticks([])
    ax_0.set_yticks([]) 
    minv, maxv = im_0.get_clim()
    ax_0.set_title(f'(EMC) XY-{test_slice}',weight='bold',fontsize=6) 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,shrink=0.351) 
    c_bar_0.set_ticks([minv,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim() 
    ax_1.set_title(f'(EMC) XZ-{test_slice}',weight='bold',fontsize=6) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_2.set_xticks([]) 
    ax_2.set_yticks([])
    minv, maxv = im_2.get_clim()
    ax_2.set_title(f'(EMC) YZ-{test_slice}',weight='bold',fontsize=6);
    
    fig_handle = plt.figure(constrained_layout = True, dpi = 200)
    fig_handle.patch.set_facecolor('gray')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)

    xy_obj = mask_pos[:,:,test_slice]
    xz_obj = mask_pos[:,test_slice,:]
    yz_obj = mask_pos[test_slice,:,:]
    
    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_0.set_xticks([])
    ax_0.set_yticks([])
    minv, maxv = im_0.get_clim()
    ax_0.set_title(f'(Pos) XY-{test_slice}',weight='bold',fontsize=6) 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,shrink=0.35) 
    c_bar_0.set_ticks([minv,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([]) 
    minv, maxv = im_1.get_clim() 
    ax_1.set_title(f'(Pos) XZ-{test_slice}',weight='bold',fontsize=6) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_2.set_xticks([]) 
    ax_2.set_yticks([])
    inv, maxv = im_2.get_clim()
    ax_2.set_title(f'(Pos) YZ-{test_slice}',weight='bold',fontsize=6);
    
    fig_handle = plt.figure(constrained_layout = True, dpi = 200)
    fig_handle.patch.set_facecolor('gray')
    spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 3)
    
    xy_obj = mask_all[:,:,test_slice]
    xz_obj = mask_all[:,test_slice,:]
    yz_obj = mask_all[test_slice,:,:]

    ax_0 = fig_handle.add_subplot(spec_handle[0,0])
    im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
    ax_0.set_xticks([])
    ax_0.set_yticks([]) 
    inv, maxv = im_0.get_clim()
    ax_0.set_title(f'(EMC+Pos) XY-{test_slice}',weight='bold',fontsize=6) 
    c_bar_0 = plt.colorbar(im_0, ax=ax_0,shrink=0.35) 
    c_bar_0.set_ticks([minv,maxv]) 

    ax_1 = fig_handle.add_subplot(spec_handle[0,1]) 
    im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_1.set_xticks([]) 
    ax_1.set_yticks([])
    minv, maxv = im_1.get_clim()
    ax_1.set_title(f'(EMC+Pos) XZ-{test_slice}',weight='bold',fontsize=6) 

    ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
    im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
    ax_2.set_xticks([])
    ax_2.set_yticks([])
    minv, maxv = im_2.get_clim()
    ax_2.set_title(f'(EMC+Pos) YZ-{test_slice}',weight='bold',fontsize=6);
    
num_pix = np.prod(frame.shape)
write_text(f'Mean intensity: {frame.mean()}\n')
write_text(f'Min intensity: {frame.min()}\n')
write_text(f'Max intensity: {frame.max()}\n')
write_text(f'Number of negative voxels: {num_pix-mask_pos.sum()}\n')
write_text(f'Fraction of positive voxels: {mask_pos.sum()}/{num_pix}\n')
write_text(f'Percentage of positive voxels: {(mask_pos.sum()/num_pix)*100}%\n')
    
e_photon_eV = 9000
lambda_photon = (h * c) / (e_photon_eV * e)
d_detector = 0.40 # for 0.30 m simulation 0.40 m or 0.50 m, or Condor 3D: 0.  , 0.3793928613638402, 0.48306051878575307
s_pixel = 800e-6

In [ ]:
if saveCorr:
    corr_model_fname = sType + pType[3:] + "_" + f"{npats}"
    corr_model = frame.copy()
    corr_model[~mask_emc] = np.nan
    with h5py.File('./' + corr_model_fname + "_corr.h5", mode="a") as handle:
        handle["intens"] = corr_model

In [ ]:
if plotAutoCorr:
    auto_corr = np.abs(fft.ifftn(fft.fftshift(frame)))
    support_autocorr = fft.fftshift(auto_corr > 0.00002)
    support_autocorr.sum()
    
    plt.figure(dpi=50)
    plt.plot(auto_corr.flatten());
    
    plt.figure(dpi=100)
    plt.imshow(support_autocorr[center,:,:], cmap='gray')
    plt.xticks([])
    plt.yticks([])
    
    plt.figure(dpi=100)
    plt.imshow(support_autocorr[:,center,:], cmap='gray')
    plt.xticks([])
    plt.yticks([])
    
    plt.figure(dpi=100)
    plt.imshow(support_autocorr[:,:,center], cmap='gray')
    plt.xticks([])
    plt.yticks([]);

dimX = frame.shape[0]
dimY = frame.shape[1]
dimZ = frame.shape[2]

pixel_num = dimX - dimX//2
theta_pixel = 0.5 * np.arctan((pixel_num*s_pixel)/d_detector)
center_to_corner = np.sqrt((pixel_num * s_pixel) ** 2 + (pixel_num * s_pixel) ** 2)
theta_max = 0.5 * np.arctan(center_to_corner / d_detector)

resolution = lambda_photon / (2.0 * np.sin(theta_pixel))
resolution_max = lambda_photon / (2.0 * np.sin(theta_max))

voxel_size = 0.5 * resolution
write_text(f'The resolution (real-space): {resolution*1e10} Å\n')
write_text(f'The maximum resolution (real-space): {resolution_max*1e10} Å\n')
write_text(f'The voxel size (real-space): {voxel_size*1e10} Å')

# Spherical support and volume fraction estimate - from "GroEL-Mediated Protein Folding: Making the Impossible, Possible"
print('Spherical support estimate!')
H_part = 14.7e-9
D_part = 13.7e-9

R_part = (15e-9)/2
R_particle_vox = R_part/voxel_size

volume_fraction_sphere = (((4/3)*pi)*(R_particle_vox**3))/(dimX*dimY*dimZ)
print(f'Support volume: {(((4/3)*pi)*(R_part**3))*1e27} nm^3')
print(f'Support volume: {(((4/3)*pi)*(R_particle_vox**3))} vox^3')
print(f'The final volume fraction is: {volume_fraction_sphere}')
print(f'The final volume percentage is: {volume_fraction_sphere*100}%')
print(f'The diameter is: {2*R_particle_vox} voxels')

support_sphere = sphere_idx(shape=frame.shape,radius=R_particle_vox,position=(dimX//2,dimY//2,dimZ//2))
vox_sphere = support_sphere.sum()
print(f'The number of object voxels: {vox_sphere}')
print(f'The number of non-object voxels: {dimX*dimY*dimZ-vox_sphere}\n')
vol_frac = volume_fraction_sphere

<h2> Performing phase retrieval</h2> 

In [ ]:
alg = "raar"

n_recons = 2
niter_alg = 1100
niter_er = 400
niter_store = 2  # number of output images

beta_start = 0.99
beta_end = beta_start

i_frac, f_frac = 1.10, 0.40
volume_i, volume_f = i_frac * vol_frac, f_frac * vol_frac

blur_i, blur_f = 1.5, 1.0

supp_update = 50

niter_store_errors = (niter_alg + niter_er) // supp_update # number of real/fourier error metric points

recon_intens = frame.copy()
recon_mask = mask_emc.copy()

def_rng = np.random.default_rng()

recon_real_array = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape), dtype=np.complex64)
recon_fourier_array = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape), dtype=np.complex64)
recon_support_array = np.zeros(shape=(n_recons, niter_store, *recon_intens.shape), dtype=np.bool)

support_size_array = np.zeros(shape=(n_recons, niter_store_errors))
error_real_array = np.zeros(shape=(n_recons, niter_store_errors))
error_fourier_array = np.zeros(shape=(n_recons, niter_store_errors))

constraints_list = ['enforce_positivity', 'enforce_real']
#constraints_list = ['enforce_real']
#constraints_list = ['enforce_positivity']
#constraints_list = []

time_now = time.localtime(time.time())
for i in range(n_recons):
    write_text(f"\rRunning reconstruction {i+1}/{n_recons}...")
    phaser = spimage.Reconstructor()
    
    phaser.set_number_of_iterations(niter_alg + niter_er)
    phaser.set_number_of_outputs_images(niter_store)
    phaser.set_number_of_outputs_scores(niter_store_errors)

    phaser.set_initial_support(support_mask=support_sphere)
    phaser.set_mask(recon_mask)
    phaser.set_intensities(recon_intens)

    phaser.append_support_algorithm('area',blur_init=blur_i,blur_final=blur_f,area_init=volume_i,area_final=volume_f,
                                    update_period=supp_update,number_of_iterations=niter_alg)
    
    phaser.append_support_algorithm('area',blur_init=blur_i,blur_final=blur_f,area_init=volume_f,area_final=volume_f,
                                    update_period=supp_update,number_of_iterations=niter_er)

    if alg == "diffmap":
        phaser.append_phasing_algorithm(
            alg,
            constraints=constraints_list,
            beta_init=beta_start,
            beta_final=beta_end,
            number_of_iterations=niter_alg,
            gamma1=-1 / beta_start,
            gamma2=(3 - beta_start) / (2 * beta_start),
        )
    else:
        phaser.append_phasing_algorithm(
            alg,
            constraints=constraints_list,
            beta_init=beta_start,
            beta_final=beta_end,
            number_of_iterations=niter_alg,
        )

    phaser.append_phasing_algorithm(
        "er", constraints=constraints_list, number_of_iterations=niter_er
    )
    
    output = phaser.reconstruct()

    real_space = output["real_space"].astype('complex64')
    fourier_space = output["fourier_space"].astype('complex64')
    support = output["support"]
    support_size = output["support_size"]
    
    error_real = output["real_error"]
    error_fourier = output["fourier_error"]
    
    recon_real_array[i] = real_space
    recon_fourier_array[i] = fourier_space
    recon_support_array[i] = support
    
    support_size_array[i] = support_size
    error_real_array[i] = error_real
    error_fourier_array[i] = error_fourier
    clear_output(wait=True)

write_text(f"\rAll {n_recons} reconstruction(s) finished!!!")

In [ ]:
saveDebug = False
rec_num = 0

cm = 'cividis'
max_v = 0.002
color_scale = 1.0

cc = center
zoom = center

fig_handle = plt.figure(constrained_layout = True, dpi = 280)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 2, ncols = 3)
plt.suptitle(f'Slices of object amplitude and phase: \nreconstruction {rec_num+1}/{n_recons} and {n_recons}/{n_recons}')

obj = np.abs(fft.fftshift(recon_real_array[rec_num][-1]))
if saveDebug:
    obj_rec = np.abs(fft.fftshift(recon_real_array[rec_num][-1]))
    np.save(f'phasing_debugging/dens_{rec_num}_b_{beta_start}_{i_frac}_{f_frac}.npy', arr=obj_rec)

xy_obj = obj.max(axis=0)[cc-zoom:cc+zoom,cc-zoom:cc+zoom]**color_scale
xz_obj = obj.max(axis=1)[cc-zoom:cc+zoom,cc-zoom:cc+zoom]**color_scale
yz_obj = obj.max(axis=2)[cc-zoom:cc+zoom,cc-zoom:cc+zoom]**color_scale

# Reconstructed electron density rec_num
ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_0.set_xticks([])
ax_0.set_yticks([])
minv, maxv = im_0.get_clim()
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05) 
c_bar_0.set_ticks([minv,maxv*0.5,maxv]) 
    
ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_1.set_xticks([]) 
ax_1.set_yticks([]) 
minv, maxv = im_1.get_clim() 
    
ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_2.set_xticks([]) 
ax_2.set_yticks([])
minv, maxv = im_2.get_clim()

# Reconstructed final electron density
obj = np.abs(fft.fftshift(recon_real_array[n_recons-1][-1]))
xy_obj = obj.max(axis=0)[cc-zoom:cc+zoom,cc-zoom:cc+zoom]**color_scale
xz_obj = obj.max(axis=1)[cc-zoom:cc+zoom,cc-zoom:cc+zoom]**color_scale
yz_obj = obj.max(axis=2)[cc-zoom:cc+zoom,cc-zoom:cc+zoom]**color_scale

ax_0 = fig_handle.add_subplot(spec_handle[1,0])
im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_0.set_xticks([])
ax_0.set_yticks([])
minv, maxv = im_0.get_clim()
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05)
c_bar_0.set_ticks([minv,maxv]) 
    
ax_1 = fig_handle.add_subplot(spec_handle[1,1])
im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_1.set_xticks([])
ax_1.set_yticks([]) 
minv, maxv = im_1.get_clim() 
    
ax_2 = fig_handle.add_subplot(spec_handle[1,2]) 
im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_2.set_xticks([]) 
ax_2.set_yticks([]) 
minv, maxv = im_2.get_clim()
if saveDebug:
    plt.savefig(f'phasing_debugging/elec_dens_{rec_num}_b_{beta_start}_{i_frac}_{f_frac}.png');

# Reconstructed supports
xy_obj = fft.fftshift(recon_support_array[rec_num][-1].max(axis=0))[cc-zoom:cc+zoom,cc-zoom:cc+zoom]
xz_obj = fft.fftshift(recon_support_array[rec_num][-1].max(axis=1))[cc-zoom:cc+zoom,cc-zoom:cc+zoom]
yz_obj = fft.fftshift(recon_support_array[rec_num][-1].max(axis=2))[cc-zoom:cc+zoom,cc-zoom:cc+zoom]
    
fig_handle = plt.figure(constrained_layout = True, dpi = 250)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 2, ncols = 3)
plt.suptitle(f'Slices of object support: \nreconstruction {rec_num+1}/{n_recons} and {n_recons}/{n_recons}')
cm = 'gray'
max_v = 1.0
    
ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_0.set_xticks([])
ax_0.set_yticks([])
    
ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_1.set_xticks([]) 
ax_1.set_yticks([])
    
ax_2 = fig_handle.add_subplot(spec_handle[0,2])
im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_2.set_xticks([]) 
ax_2.set_yticks([])

xy_obj = fft.fftshift(recon_support_array[-1][-1].max(axis=0))[cc-zoom:cc+zoom,cc-zoom:cc+zoom]
xz_obj = fft.fftshift(recon_support_array[-1][-1].max(axis=1))[cc-zoom:cc+zoom,cc-zoom:cc+zoom]
yz_obj = fft.fftshift(recon_support_array[-1][-1].max(axis=2))[cc-zoom:cc+zoom,cc-zoom:cc+zoom]

ax_3 = fig_handle.add_subplot(spec_handle[1,0])
im_3 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_3.set_xticks([])
ax_3.set_yticks([]) 
    
ax_4 = fig_handle.add_subplot(spec_handle[1,1])
im_4 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_4.set_xticks([])
ax_4.set_yticks([]) 
    
ax_5 = fig_handle.add_subplot(spec_handle[1,2])
im_5 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_5.set_xticks([]) 
ax_5.set_yticks([])
write_text(f'{recon_support_array[:,-1].sum(axis=(1,2,3))}')
if saveDebug:
    plt.savefig(f'phasing_debugging/supports_{rec_num}_b_{beta_start}_{i_frac}_{f_frac}.png');

# Reconstructed Fourier intensities
fig_handle = plt.figure(constrained_layout = True, dpi = 250)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 2, ncols = 3)
plt.suptitle(f'Slices of recovered Fourier intensity: \nreconstruction {rec_num+1}/{n_recons}')
cm = 'viridis'
max_v = 0.1

im_slice = dimX//2
recon_fourier = np.abs(recon_fourier_array[rec_num][-1]) ** 2
fourier_scaling = 1.0

yz_obj = recon_fourier[:,:,im_slice] ** fourier_scaling
xz_obj = recon_fourier[:,im_slice,:] ** fourier_scaling
xy_obj = recon_fourier[im_slice,:,:] ** fourier_scaling

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
im_0 = plt.imshow(xy_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None) 
ax_0.set_xticks([])
ax_0.set_yticks([]) 
minv, maxv = im_0.get_clim()
ax_0.set_title(f'(XY-{im_slice}/{dimX})',weight='bold',fontsize=6) 
c_bar_0 = plt.colorbar(im_0, ax=ax_0,fraction=0.05) 
c_bar_0.set_ticks([minv,maxv*0.5,maxv]) 

ax_1 = fig_handle.add_subplot(spec_handle[0,1])
im_1 = plt.imshow(xz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_1.set_xticks([]) 
ax_1.set_yticks([])
minv, maxv = im_1.get_clim()
ax_1.set_title(f'(XZ-{im_slice}/{dimY})',weight='bold',fontsize=6) 

ax_2 = fig_handle.add_subplot(spec_handle[0,2]) 
im_2 = plt.imshow(yz_obj,vmin=0.0,vmax=max_v,cmap=cm,interpolation=None)
ax_2.set_xticks([]) 
ax_2.set_yticks([])
minv, maxv = im_2.get_clim()
ax_2.set_title(f'(YZ-{im_slice}/{dimZ})',weight='bold',fontsize=6);
if saveDebug:
    plt.savefig(f'phasing_debugging/fourier_intens_{rec_num}_b_{beta_start}_{i_frac}_{f_frac}.png');

fig_handle = plt.figure(constrained_layout = True, dpi = 280)
fig_handle.patch.set_facecolor(f'white')
spec_handle = fig_handle.add_gridspec(nrows = 1, ncols = 2)

iter_vec = np.linspace(1, niter_alg+niter_er, num = niter_store_errors-1)

ax_0 = fig_handle.add_subplot(spec_handle[0,0])
plt_0 = plt.plot(iter_vec, error_real_array[:,1:].T,'b')
plt.yscale('log')
plt.xlabel('iteration #', weight='bold')
plt.ylabel('error', weight='bold')
plt.title('Real-space error', weight='bold')
    
ax_1 = fig_handle.add_subplot(spec_handle[0,1])
splt_1 = plt.plot(iter_vec, error_fourier_array[:,1:].T,'r');
plt.yscale('log')
plt.xlabel('iteration #', weight='bold')
plt.ylabel('error', weight='bold')
plt.title('Fourier-space error', weight='bold');
if saveDebug:
    plt.savefig(f'phasing_debugging/errors_{rec_num}_b_{beta_start}_{i_frac}_{f_frac}.png');

In [ ]:
recs_rs = recon_real_array[:,-1,:,:,:]
sups = recon_support_array[:,-1,:,:,:]
prtf_res = spimage.prtf(recs_rs, sups)
prtf_r = spimage.radial(prtf_res['prtf'])

In [ ]:
av_dens = prtf_res['super_image']
test_crop = center
plt.figure()
plt.imshow(np.abs(fft.fftshift(av_dens)).max(axis=0)[cc-test_crop:cc+test_crop,cc-test_crop:cc+test_crop])
plt.xticks([])
plt.yticks([])
plt.colorbar()
plt.figure()
plt.imshow(np.abs(fft.fftshift(av_dens)).max(axis=1)[cc-test_crop:cc+test_crop,cc-test_crop:cc+test_crop])
plt.xticks([])
plt.yticks([])
plt.colorbar()
plt.figure()
plt.imshow(np.abs(fft.fftshift(av_dens)).max(axis=2)[cc-test_crop:cc+test_crop,cc-test_crop:cc+test_crop])
plt.xticks([])
plt.yticks([])
plt.colorbar();
#plt.savefig('dens_prot_wat_0_100k_1M_bg_beta_010');

In [ ]:
pix_emc = 1 / (dimX * voxel_size * 1e9) # in nm^-1
max_points = prtf_r.shape[0]
fp_resolution_r_inv = np.arange(0, max_points) * pix_emc
xcp, _ = interpolated_intercepts(fp_resolution_r_inv, prtf_r, np.repeat(1/np.exp(1), max_points))
print(f'{xcp} nm^-1 and {1/xcp} nm')
plt.figure()
plt.plot(fp_resolution_r_inv, prtf_r)
plt.axhline(y=1/np.exp(1), xmin=0.0, xmax=1.0)
plt.xlabel('$|\mathbf{q}|$ in $nm^{-1}$')
plt.ylabel('PRTF', weight='bold');
#plt.savefig('prtf_prot_wat_0_100k_1M_bg_beta_010');

In [ ]:
s_image_abs = np.abs(av_dens)
s_image_abs_shifted = fft.fftshift(s_image_abs)

s_support = s_image_abs_shifted > (0.06 * s_image_abs_shifted.max())
conv_hull_supp = convex_hull_image(s_support)

prtf_im = prtf_res['prtf']
prtf_im_ft = fft.ifftn((prtf_im))
prtf_im_smooth = np.abs(fft.fftn(prtf_im_ft * fft.fftshift(conv_hull_supp)))
prtf_r_smooth = spimage.radial(prtf_im_smooth)

plt.plot(fp_resolution_r_inv, prtf_r, 'g');
plt.plot(fp_resolution_r_inv, prtf_r_smooth, 'r');

print(np.trapz(prtf_r))
print(np.trapz(prtf_r_smooth))
print(np.abs(np.trapz(prtf_r)-np.trapz(prtf_r_smooth)))